## Pluvial Floods Model V2

What's different from V1?

Expanded coverage from Montréal only to **Montréal + Laval**.

- Pluvial flood dates are the same (events affect both cities similarly)
- Daily meteorological data is now averaged across **both Montréal and Laval weather stations**
- The model has been retrained on the new averaged data
- Fluvial (river) flood tracking removed — this notebook focuses exclusively on pluvial floods

---
### 1. Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import joblib
from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report, roc_auc_score, average_precision_score,
    f1_score, precision_score, recall_score,
    precision_recall_curve, confusion_matrix
)

---
### 2. Build Dataset

Load historical station CSVs, average across Montréal and Laval stations, label pluvial flood dates, and save.

In [2]:
years = range(2015, 2027)

numeric_cols = [
    "Temp max.(°C)",
    "Temp min.(°C)",
    "Temp moy.(°C)",
    "Précip. tot. (mm)",
    "Neige au sol (cm)"
]

columns_to_keep = ["Date/Heure"] + numeric_cols

# Stations grouped by city — each city gets equal weight in the final average
stations_by_city = {
    "montreal": [
        "702S006",  # Montreal-Trudeau
        "7025251",  # MONTREAL INTL A
    ],
    "laval": [
        "7024745",  # LAVAL
        "7024627",  # LAVAL DES RAPIDES
    ],
}

df_list = []

for year in years:
    city_means = []

    for city, stations in stations_by_city.items():
        station_dfs = []

        for station in stations:
            file_path = f"/weather_data/yearly_data/fr_climat_quotidiennes_QC_{station}_{year}_P1D.csv"
            if not os.path.exists(file_path):
                continue

            df = pd.read_csv(file_path)[columns_to_keep]
            df[numeric_cols] = (
                df[numeric_cols]
                .astype(str)
                .apply(lambda col: col.str.replace(",", ".", regex=False))
                .apply(pd.to_numeric, errors="coerce")
            )
            station_dfs.append(df)

        if station_dfs:
            city_combined = pd.concat(station_dfs)
            city_mean = city_combined.groupby("Date/Heure", as_index=False)[numeric_cols].mean()
            city_mean["city"] = city
            city_means.append(city_mean)

    if city_means:
        all_cities = pd.concat(city_means)
        averaged = all_cities.groupby("Date/Heure", as_index=False)[numeric_cols].mean().round(1)
        df_list.append(averaged)

weather_df = pd.concat(df_list, ignore_index=True)

# Date processing
weather_df["Date/Heure"] = pd.to_datetime(weather_df["Date/Heure"])
weather_df = weather_df.sort_values("Date/Heure").reset_index(drop=True)
weather_df = weather_df[
    weather_df["Date/Heure"] <= pd.Timestamp.today().normalize() - pd.Timedelta(days=1)
]

# Interpolate missing values
cols_to_fill = ["Temp moy.(°C)", "Précip. tot. (mm)"]
weather_df[cols_to_fill] = weather_df[cols_to_fill].interpolate(method="linear", limit_direction="both")

# Pluvial flood dates — same events affect Montréal and Laval
pluvial_dates = [
    "2016-08-16", "2019-10-01", "2019-10-17", "2019-10-31",
    "2022-06-16", "2022-09-13", "2023-07-13", "2023-10-07",
    "2024-07-10", "2024-08-09", "2025-07-13"
]

weather_df["Inondation pluviale"] = (
    weather_df["Date/Heure"].isin(pd.to_datetime(pluvial_dates)).astype(int)
)

output_path = "/weather_data/weather_with_floods_2015_2026.csv"
weather_df.to_csv(output_path, index=False)

print(f"Saved to {output_path}")
print(f"Shape: {weather_df.shape}")
print(f"Pluvial flood days: {weather_df['Inondation pluviale'].sum()}")

ValueError: No objects to concatenate

---
### 3. Exploratory Analysis

Read the CSV, engineer rolling features, and plot each variable against flood events.

In [ ]:
file_path = "../../server/app/data/weather_with_floods_2015_2026.csv"
weather_df = pd.read_csv(file_path)
weather_df["Date/Heure"] = pd.to_datetime(weather_df["Date/Heure"])

print("Shape:", weather_df.shape)
print("\nMissing values:")
print(weather_df[["Temp moy.(°C)", "Précip. tot. (mm)"]].isna().sum())

In [ ]:
# Rolling rain and derived features
weather_df["Rain_1d"] = weather_df["Précip. tot. (mm)"]
weather_df["Rain_3d"] = weather_df["Précip. tot. (mm)"].rolling(3).sum()
weather_df["Rain_5d"] = weather_df["Précip. tot. (mm)"].rolling(5).sum()
weather_df["Rain_7d"] = weather_df["Précip. tot. (mm)"].rolling(7).sum()
weather_df["Rain_intensity"] = weather_df["Rain_1d"] / (weather_df["Rain_7d"] + 1)
weather_df["Temp_pos"] = weather_df["Temp moy.(°C)"].clip(lower=0)
weather_df["Melt_Index"] = weather_df["Temp_pos"] * weather_df["Neige au sol (cm)"]
weather_df["Melt_3d"] = weather_df["Melt_Index"].rolling(3).sum()
weather_df["Temp_diff_2d"] = weather_df["Temp moy.(°C)"] - weather_df["Temp moy.(°C)"].shift(2)

In [ ]:
def plot_variable_with_floods(df, variable_name):
    plot_df = df.copy()
    min_val = plot_df[variable_name].min()
    max_val = plot_df[variable_name].max()
    margin = (max_val - min_val) * 0.05

    plt.figure(figsize=(12, 5))

    rain_vars = ["Précip. tot. (mm)", "Rain_1d", "Rain_3d", "Rain_5d", "Rain_7d", "Rain_intensity"]
    if variable_name in rain_vars:
        plot_df = plot_df[plot_df["Temp moy.(°C)"] > 0]  # exclude frozen precipitation
        plt.bar(plot_df["Date/Heure"], plot_df[variable_name], label=variable_name, width=10)
        plt.ylim(min_val, max_val + margin)
    else:
        plt.plot(plot_df["Date/Heure"], plot_df[variable_name], label=variable_name)
        plt.ylim(min_val - margin, max_val + margin)

    plt.scatter(
        plot_df.loc[plot_df["Inondation pluviale"] == 1, "Date/Heure"],
        plot_df.loc[plot_df["Inondation pluviale"] == 1, variable_name],
        color="red", label="Inondation pluviale", zorder=5
    )

    plt.xlabel("Date/Heure")
    plt.ylabel(variable_name)
    plt.title(f"{variable_name} — Montréal & Laval (2015–2026)")
    plt.xticks(rotation=45)
    plt.legend()
    plt.tight_layout()
    plt.show()


for var in ["Précip. tot. (mm)", "Rain_intensity", "Temp moy.(°C)",
            "Rain_1d", "Rain_3d", "Rain_5d", "Rain_7d", "Melt_3d", "Temp_diff_2d"]:
    plot_variable_with_floods(weather_df, var)

In [ ]:
# Top 20 highest precipitation days
top_precip = weather_df.sort_values("Précip. tot. (mm)", ascending=False)
print(top_precip[["Date/Heure", "Précip. tot. (mm)", "Inondation pluviale"]].head(20))

---
### 4. Preprocessing

Feature engineering and chronological train/test split.

In [ ]:
file_path = "../../server/app/data/weather_with_floods_2015_2026.csv"
weather_df = pd.read_csv(file_path)
weather_df["Date/Heure"] = pd.to_datetime(weather_df["Date/Heure"])

df = weather_df.sort_values("Date/Heure").reset_index(drop=True)

# Season
df["Month"] = df["Date/Heure"].dt.month
def get_season(month):
    if month in [12, 1, 2]: return "winter"
    if month in [3, 4, 5]: return "spring"
    if month in [6, 7, 8]: return "summer"
    return "fall"
df["Season"] = df["Month"].apply(get_season)

# Rolling rain features
df["rain_1d"] = df["Précip. tot. (mm)"]
df["rain_3d"] = df["Précip. tot. (mm)"].rolling(3, min_periods=1).sum()
df["rain_5d"] = df["Précip. tot. (mm)"].rolling(5, min_periods=1).sum()
df["rain_7d"] = df["Précip. tot. (mm)"].rolling(7, min_periods=1).sum()
df["rain_intensity"] = df["rain_1d"] / (df["rain_7d"] + 1)
df["is_freezing"] = (df["Temp moy.(°C)"] <= 0).astype(int)
df["Temp_diff_2d"] = df["Temp moy.(°C)"].diff(2)
df = df.dropna(subset=["Temp_diff_2d"])

features = [
    "Temp moy.(°C)", "rain_1d", "rain_3d", "rain_5d", "rain_7d",
    "Temp_diff_2d", "Season", "rain_intensity", "is_freezing"
]
X = df[features]
y = df["Inondation pluviale"]

# Chronological 80/20 split
split_index = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

X_train = pd.get_dummies(X_train, columns=["Season"], drop_first=True)
X_test  = pd.get_dummies(X_test,  columns=["Season"], drop_first=True)
X_test  = X_test.reindex(columns=X_train.columns, fill_value=0)

# Full dataset (for final training after CV)
X_full = pd.get_dummies(X, columns=["Season"], drop_first=True)
y_full = y

print(f"Train: {X_train.shape}, flood days: {y_train.sum()}")
print(f"Test:  {X_test.shape},  flood days: {y_test.sum()}")

---
### 5. Cross-Validation

Stratified time-series CV to evaluate XGBoost before final training.

In [ ]:
class StratifiedTimeSeriesSplit:
    def __init__(self, n_splits, min_train_pos=3, min_test_pos=3):
        self.n_splits = n_splits
        self.min_train_pos = min_train_pos
        self.min_test_pos = min_test_pos

    def split(self, y):  # Expanding window folds
        y = np.array(y)
        n_samples = len(y)
        fold_size = n_samples // (self.n_splits + 1)

        for i in range(self.n_splits):
            train_end = (i + 1) * fold_size
            test_end = train_end + fold_size if i < self.n_splits - 1 else n_samples

            while train_end < n_samples and np.sum(y[:train_end]) < self.min_train_pos:
                train_end += 1
            while test_end < n_samples and np.sum(y[train_end:test_end]) < self.min_test_pos:
                test_end += 1

            if train_end >= test_end:
                break

            train_idx = np.arange(0, train_end)
            test_idx  = np.arange(train_end, test_end)

            if np.sum(y[test_idx]) == 0:
                future_pos = np.where(y[train_end:] == 1)[0]
                if len(future_pos) > 0:
                    test_idx = np.arange(train_end, future_pos[0] + train_end + 1)

            yield train_idx, test_idx


In [ ]:
n_splits = 4
tscv = StratifiedTimeSeriesSplit(n_splits=n_splits)

xgb = XGBClassifier(
    n_estimators=900, max_depth=12, learning_rate=0.0970430828887202,
    subsample=0.5989281618911819, colsample_bytree=0.8738047115704085,
    min_child_weight=10, gamma=0.5652786020709911,
    scale_pos_weight=len(y_full[y_full==0]) / len(y_full[y_full==1]),
    random_state=42, eval_metric="auc", n_jobs=-1
)

cv_results = {"f1": [], "roc_auc": [], "pr_auc": [], "threshold": []}

for fold, (train_idx, test_idx) in enumerate(tscv.split(y_full)):
    X_tr, X_te = X_full.iloc[train_idx], X_full.iloc[test_idx]
    y_tr, y_te = y_full.iloc[train_idx], y_full.iloc[test_idx]

    xgb.fit(X_tr, y_tr)
    proba = xgb.predict_proba(X_te)[:, 1]

    thresholds = np.linspace(0, 1, 100)
    f1s = [f1_score(y_te, (proba >= t).astype(int), zero_division=0) for t in thresholds]
    best_t = thresholds[np.argmax(f1s)]

    cv_results["f1"].append(max(f1s))
    cv_results["roc_auc"].append(roc_auc_score(y_te, proba))
    cv_results["pr_auc"].append(average_precision_score(y_te, proba))
    cv_results["threshold"].append(best_t)

    print(f"Fold {fold+1} — F1: {max(f1s):.3f} | ROC-AUC: {cv_results['roc_auc'][-1]:.3f} "
          f"| PR-AUC: {cv_results['pr_auc'][-1]:.3f} | Best threshold: {best_t:.2f}")

print(f"\nMean F1:      {np.mean(cv_results['f1']):.3f}")
print(f"Mean ROC-AUC: {np.mean(cv_results['roc_auc']):.3f}")
print(f"Mean PR-AUC:  {np.mean(cv_results['pr_auc']):.3f}")
avg_threshold = np.mean(cv_results['threshold'])
print(f"Avg threshold: {avg_threshold:.3f}")

---
### 6. Hyperparameter Tuning (Optuna)

Run only when re-tuning is needed. Uses stratified time-series CV and PR-AUC as objective.

In [ ]:
import optuna
from optuna.pruners import MedianPruner

def objective_xgb(trial):
    params = {
        "n_estimators":      trial.suggest_int("n_estimators", 300, 1000, step=50),
        "max_depth":         trial.suggest_int("max_depth", 3, 15),
        "learning_rate":     trial.suggest_float("learning_rate", 0.01, 0.3),
        "subsample":         trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight":  trial.suggest_int("min_child_weight", 1, 10),
        "gamma":             trial.suggest_float("gamma", 0, 5),
    }
    model = XGBClassifier(
        **params,
        scale_pos_weight=len(y_full[y_full==0]) / len(y_full[y_full==1]),
        random_state=42, eval_metric="auc", n_jobs=-1
    )
    pr_aucs = []
    for train_idx, test_idx in tscv.split(y_full):
        model.fit(X_full.iloc[train_idx], y_full.iloc[train_idx])
        proba = model.predict_proba(X_full.iloc[test_idx])[:, 1]
        pr_aucs.append(average_precision_score(y_full.iloc[test_idx], proba))
        trial.report(np.mean(pr_aucs), step=len(pr_aucs))
        if trial.should_prune():
            raise optuna.TrialPruned()
    return np.mean(pr_aucs)

study = optuna.create_study(direction="maximize", pruner=MedianPruner(n_warmup_steps=5))
study.optimize(objective_xgb, n_trials=500, show_progress_bar=True)

print("Best params:", study.best_params)
print("Best PR-AUC:", study.best_value)

---
### 7. Final Evaluation

Train on the full dataset, evaluate on the held-out 20% test set.

In [ ]:
xgb_final = XGBClassifier(
    n_estimators=900, max_depth=12, learning_rate=0.0970430828887202,
    subsample=0.5989281618911819, colsample_bytree=0.8738047115704085,
    min_child_weight=10, gamma=0.5652786020709911,
    scale_pos_weight=len(y_train[y_train==0]) / len(y_train[y_train==1]),
    random_state=42, eval_metric="auc", n_jobs=-1
)
xgb_final.fit(X_train, y_train)
y_proba = xgb_final.predict_proba(X_test)[:, 1]
y_pred  = (y_proba >= avg_threshold).astype(int)

print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.3f}")
print(f"PR-AUC:  {average_precision_score(y_test, y_proba):.3f}")
print()
print(classification_report(y_test, y_pred, target_names=["No Flood", "Pluvial Flood"]))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

In [ ]:
# Precision & Recall vs Threshold
thresholds = np.linspace(0, 1, 100)
prec_scores = [precision_score(y_test, (y_proba >= t).astype(int), zero_division=0) for t in thresholds]
rec_scores  = [recall_score(y_test,    (y_proba >= t).astype(int), zero_division=0) for t in thresholds]

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(thresholds, prec_scores, label="Precision")
plt.plot(thresholds, rec_scores,  label="Recall")
plt.axvline(avg_threshold, color="gray", linestyle="--", label=f"Threshold ({avg_threshold:.2f})")
plt.title("Precision & Recall vs Threshold")
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.legend()
plt.grid(True)

precision, recall, _ = precision_recall_curve(y_test, y_proba)
plt.subplot(1, 2, 2)
plt.plot(recall, precision)
plt.title("Precision-Recall Curve")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Feature importance
from xgboost import plot_importance
plot_importance(xgb_final, importance_type="gain", max_num_features=15)
plt.tight_layout()
plt.show()

---
### 8. Save & Load Model

In [ ]:
# Train on full data before saving
xgb_full = XGBClassifier(
    n_estimators=900, max_depth=12, learning_rate=0.0970430828887202,
    subsample=0.5989281618911819, colsample_bytree=0.8738047115704085,
    min_child_weight=10, gamma=0.5652786020709911,
    scale_pos_weight=len(y_full[y_full==0]) / len(y_full[y_full==1]),
    random_state=42, eval_metric="auc", n_jobs=-1
)
xgb_full.fit(X_full, y_full)

joblib.dump(xgb_full, "xgb_pluvial_flood_model_v2.pkl")
print("Model saved: xgb_pluvial_flood_model_v2.pkl")

In [ ]:
# Load and verify
xgb_loaded = joblib.load("xgb_pluvial_flood_model_v2.pkl")
sample_proba = xgb_loaded.predict_proba(X_test[:5])[:, 1]
print("Sample probabilities:", sample_proba.round(3))